# Part 4: Multi-Agent Orchestration — SOC Edition

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

# Auto-detect pre-provisioned environment
config_path = Path.cwd().parent / "workshop_config.json"
if not config_path.exists():
    config_path = Path.cwd() / "workshop_config.json"
if config_path.exists():
    _cfg = json.loads(config_path.read_text())
    PROJECT_ENDPOINT = _cfg["PROJECT_ENDPOINT"]
    MODEL_DEPLOYMENT_NAME = _cfg["MODEL_DEPLOYMENT_NAME"]
else:
    raise FileNotFoundError(
        "workshop_config.json not found. Run 00-setup.ipynb first."
    )

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo
   Model: gpt-4.1-mini


---
## Section 5: Multi-Agent Orchestration (~15 min)

The **Agent Framework SDK** (`agent-framework`) provides orchestration patterns
for coordinating multiple agents. These agents run **in-memory** — ideal for
prototyping complex SOC workflows.

| Pattern | Description | SOC Use Case |
|---------|-------------|-------------|
| **Sequential** | Agent A → Agent B → Agent C | Alert triage → enrichment → report generation |
| **Concurrent** | Fan-out / fan-in with aggregation | Parallel risk factor evaluation |
| **Handoff** | Router delegates to specialists | Triage routes to compromise/phishing/insider threat specialists |
| **Group Chat** | Agents discuss collaboratively | Multi-analyst review of complex incidents |

> Agent Framework automatically emits OpenTelemetry spans — visible in Foundry portal → Tracing.

In [3]:
# --- 5.0 Setup Agent Framework ---
import asyncio
from dataclasses import dataclass
from typing import Annotated
from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    AgentResponse,
    Case,
    Default,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.foundry import FoundryChatClient, FoundryAgent
from agent_framework.orchestrations import (
    HandoffBuilder,
    GroupChatBuilder,
)
from azure.ai.projects.models import MCPTool, WebSearchPreviewTool
from typing_extensions import Never
from agent_framework import InMemoryCheckpointStorage

# Create a shared chat client connected to the Foundry project
af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=AzureCliCredential(),
)

print("✅ Agent Framework client ready")

✅ Agent Framework client ready


In [4]:
# --- 5.1 Sequential Orchestration: Alert Triage Pipeline ---
# Triager → Enricher → Report Writer: classify the alert, enrich with context, then write a report.
# chain_only_agent_responses=True: each agent sees only the previous agent's output.

from agent_framework import AgentResponseUpdate

print("=" * 60)
print("SEQUENTIAL: Triager → Enricher → Report Writer")
print("=" * 60)

seq_triager = Agent(
    client=af_client, name="Triager",
    instructions=(
        "You are a SOC L1 triage analyst. Given a security alert description, "
        "classify it by type (Impossible Travel, Brute Force, MFA Fatigue, OAuth Phishing, etc.), "
        "assign a severity (Critical/High/Medium/Low), and list the key indicators of compromise. "
        "Keep output concise — under 150 words."
    ),
)

seq_enricher = Agent(
    client=af_client, name="Enricher",
    instructions=(
        "You are a SOC enrichment specialist. Given a triaged alert, add context: "
        "map to MITRE ATT&CK techniques, estimate the travel velocity if locations are given, "
        "assess IP reputation indicators, and note any privilege escalation risk. "
        "Output only the enriched context — under 200 words."
    ),
)

seq_reporter = Agent(
    client=af_client, name="ReportWriter",
    instructions=(
        "You are a SOC report writer. Given enriched alert context, produce a concise "
        "executive-ready incident report with: Summary, Risk Level, MITRE Techniques, "
        "Verdict (PACO framework), and Recommended Actions. Use markdown formatting. "
        "Keep under 300 words."
    ),
)

seq_workflow = (
    WorkflowBuilder(name="SOC Alert Triage Pipeline", start_executor=seq_triager)
    .add_chain([seq_triager, seq_enricher, seq_reporter])
    .build()
)

# Run with streaming to see each agent's output as it arrives
last_agent = None
async for event in seq_workflow.run(
    "Alert: user bob.garcia@contoso.com signed in from Denver, US at 03:10 and Beijing, CN 20 minutes later. "
    "First sign-in used password+MFA from corporate IP 10.0.0.5. Second sign-in was password-only from IP 45.33.32.156 "
    "(known Tor exit node). User holds Exchange Administrator role.",
    stream=True,
):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        if event.data.author_name != last_agent:
            last_agent = event.data.author_name
            print(f"\n\n[{last_agent}]:", end=" ")
        print(event.data.text, end="", flush=True)

print("\n\n✅ Sequential orchestration complete (Triager → Enricher → Report Writer)")

SEQUENTIAL: Triager → Enricher → Report Writer


C:\Users\haozhang2\AppData\Local\Temp\ipykernel_24928\2818287308.py:44: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()




[Triager]: Type: Impossible Travel  
Severity: High  
Indicators of Compromise:  
- User logged in from geographically distant locations (Denver to Beijing) within a 20-minute window, which is practically impossible.  
- First login used password+MFA from a corporate IP (legitimate).  
- Second login used password-only from a known Tor exit node IP—indicating possible credential compromise and bypass of MFA.  
- User has Exchange Administrator role, increasing potential impact.

[Enricher]: Mapped MITRE ATT&CK Techniques:  
- T1078: Valid Accounts (use of legitimate credentials)  
- T1110: Brute Force or Credential Access (password-only login from Tor node)  
- T1078.003: Valid Accounts: Cloud Accounts (Exchange Admin access)  
- T1136: Create or Modify Account (potential lateral movement or escalation)

Travel Velocity:  
- Denver (39.7392°N, 104.9903°W) to Beijing (39.9042°N, 116.4074°E) ≈ 10,000 km in 20 minutes  
- Required speed ≈ 30,000 km/h, well beyond commercial or private f

In [5]:
# --- 5.1b Sequential with FoundryAgent (server-side agents → traces in portal) ---
# Same Triager → Enricher → ReportWriter workflow, but using FoundryAgent backed by
# Prompt Agents registered in Foundry. Each agent invocation produces server-side traces.

# Register the agents as Foundry Prompt Agents
triager_prompt = project_client.agents.create_version(
    agent_name="04-soc-seq-triager",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC L1 triage analyst. Classify the alert by type, assign severity, "
            "and list key indicators of compromise. Keep output concise."
        ),
    ),
)
print(f"✅ Foundry agent: {triager_prompt.name} v{triager_prompt.version}")

enricher_prompt = project_client.agents.create_version(
    agent_name="04-soc-seq-enricher",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC enrichment specialist. Add MITRE ATT&CK mappings, travel velocity estimates, "
            "IP reputation assessment, and privilege escalation risk."
        ),
    ),
)
print(f"✅ Foundry agent: {enricher_prompt.name} v{enricher_prompt.version}")

reporter_prompt = project_client.agents.create_version(
    agent_name="04-soc-seq-reporter",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC report writer. Produce an executive-ready incident report with "
            "Summary, Risk Level, MITRE Techniques, PACO Verdict, and Recommended Actions."
        ),
    ),
)
print(f"✅ Foundry agent: {reporter_prompt.name} v{reporter_prompt.version}")

# Wrap as FoundryAgent instances
foundry_triager = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=triager_prompt.name,
    agent_version=triager_prompt.version, credential=AzureCliCredential(), name="Triager",
)
foundry_enricher = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=enricher_prompt.name,
    agent_version=enricher_prompt.version, credential=AzureCliCredential(), name="Enricher",
)
foundry_reporter = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT, agent_name=reporter_prompt.name,
    agent_version=reporter_prompt.version, credential=AzureCliCredential(), name="ReportWriter",
)

# Build sequential workflow with FoundryAgent participants
print("\n" + "=" * 60)
print("SEQUENTIAL (FoundryAgent): Triager → Enricher → ReportWriter")
print("=" * 60)

seq_foundry = (
    WorkflowBuilder(name="SOC Sequential FoundryAgent", start_executor=foundry_triager)
    .add_chain([foundry_triager, foundry_enricher, foundry_reporter])
    .build()
)

last_agent = None
async for event in seq_foundry.run(
    "Alert: user bob.garcia@contoso.com signed in from Denver, US at 03:10 and Beijing, CN 20 minutes later. "
    "Second sign-in from Tor exit node. User is Exchange Administrator.",
    stream=True,
):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        if event.data.author_name != last_agent:
            last_agent = event.data.author_name
            print(f"\n\n[{last_agent}]:", end=" ")
        print(event.data.text, end="", flush=True)

print("\n\n✅ Sequential (FoundryAgent) complete!")
print("   Check Foundry Portal → Tracing for server-side traces.")

✅ Foundry agent: 04-soc-seq-triager v1
✅ Foundry agent: 04-soc-seq-enricher v1
✅ Foundry agent: 04-soc-seq-reporter v1

SEQUENTIAL (FoundryAgent): Triager → Enricher → ReportWriter


C:\Users\haozhang2\AppData\Local\Temp\ipykernel_24928\4005865556.py:64: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()




[Triager]: - **Alert Type:** Suspicious login / Possible account compromise  
- **Severity:** High  
- **Key Indicators of Compromise (IOCs):**  
  - Rapid geographically impossible login (Denver to Beijing in 20 minutes)  
  - Second sign-in from Tor exit node (likely anonymizing connection)  
  - Target user is Exchange Administrator (high-privilege account)

[Enricher]: ### MITRE ATT&CK Mapping  
| Technique ID   | Technique Name                     | Description                                        |
|----------------|----------------------------------|----------------------------------------------------|
| T1078          | Valid Accounts                   | Use of legitimate credentials to gain access     |
| T1090.003      | Proxy: Tor                      | Use of Tor network to anonymize connection        |
| T1531          | Account Access Removal          | Possible unauthorized use of admin account        |
| T1201          | Password Policy Discovery       | Attackers m

In [6]:
# --- 5.2 Concurrent Fan-Out / Fan-In: Parallel Risk Assessment ---
# A dispatcher fans out the alert to 3 specialist agents in parallel,
# then an aggregator joins their assessments into a consolidated risk report.

print("=" * 60)
print("CONCURRENT: Dispatcher → [IdentityAnalyst + ThreatIntelAnalyst + ComplianceAnalyst] → Aggregator")
print("=" * 60)

# --- Custom Executors ---

class DispatchToAnalysts(Executor):
    """Fans out the incoming alert to all specialist agents in parallel."""
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        initial_message = Message("user", contents=[prompt])
        await ctx.send_message(AgentExecutorRequest(messages=[initial_message], should_respond=True))


@dataclass
class AggregatedAssessment:
    identity_analysis: str
    threat_intel: str
    compliance: str


class AggregateAssessments(Executor):
    """Joins specialist assessments into a single consolidated risk report."""
    @handler
    async def aggregate(self, results: list[AgentExecutorResponse], ctx: WorkflowContext[Never, str]) -> None:
        by_id: dict[str, str] = {}
        for r in results:
            by_id[r.executor_id] = r.agent_response.text
        consolidated = (
            f"📊 Consolidated Risk Assessment\n{'=' * 40}\n\n"
            f"🔐 Identity Analysis:\n{by_id.get('identity_analyst', '')}\n\n"
            f"🕵️ Threat Intelligence:\n{by_id.get('threat_intel_analyst', '')}\n\n"
            f"📋 Compliance & Policy:\n{by_id.get('compliance_analyst', '')}\n"
        )
        await ctx.yield_output(consolidated)


# --- Build the workflow ---

dispatcher = DispatchToAnalysts(id="dispatcher")
aggregator = AggregateAssessments(id="aggregator")

expert_identity = AgentExecutor(
    Agent(client=af_client, name="identity_analyst",
          instructions=(
              "You are an identity security analyst. Evaluate the alert for: authentication anomalies, "
              "MFA status, device compliance, session tokens, and user risk state. "
              "Provide a concise risk assessment under 150 words."
          ))
)
expert_threat_intel = AgentExecutor(
    Agent(client=af_client, name="threat_intel_analyst",
          instructions=(
              "You are a threat intelligence analyst. Evaluate IP reputation, geolocation anomalies, "
              "known threat actor TTPs, and MITRE ATT&CK mapping. "
              "Provide a concise assessment under 150 words."
          ))
)
expert_compliance = AgentExecutor(
    Agent(client=af_client, name="compliance_analyst",
          instructions=(
              "You are a compliance and policy analyst. Assess the alert against organizational "
              "security policies: SLA adherence, data access risk, regulatory implications (GDPR, SOX), "
              "and required escalation paths. Provide a concise assessment under 150 words."
          ))
)

conc_workflow = (
    WorkflowBuilder(name="SOC Parallel Risk Assessment", start_executor=dispatcher)
    .add_fan_out_edges(dispatcher, [expert_identity, expert_threat_intel, expert_compliance])
    .add_fan_in_edges([expert_identity, expert_threat_intel, expert_compliance], aggregator)
    .build()
)

# Run the workflow
async for event in conc_workflow.run(
    "Alert: user emma.jones@contoso.com (VP of Engineering) signed in from San Francisco at 05:20 "
    "and Mumbai, India 15 minutes later. Second sign-in from IP 45.77.88.99 using password-only auth. "
    "Immediately after, an OAuth app 'DevTools-Pro' was granted Directory.Read.All permissions.",
    stream=True,
):
    if event.type == "output" and event.executor_id == "aggregator":
        print(event.data)

print("\n✅ Concurrent fan-out/fan-in complete")

CONCURRENT: Dispatcher → [IdentityAnalyst + ThreatIntelAnalyst + ComplianceAnalyst] → Aggregator


C:\Users\haozhang2\AppData\Local\Temp\ipykernel_24928\2435061564.py:76: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()


📊 Consolidated Risk Assessment

🔐 Identity Analysis:
The alert indicates high-risk behavior: two geographically impossible sign-ins within 15 minutes suggest credential compromise or session token misuse. The second sign-in lacked MFA, increasing vulnerability. Device compliance status is unknown, but the password-only authentication from Mumbai is a red flag. Granting an OAuth app broad Directory.Read.All permissions immediately after suggests potential malicious lateral movement or data exfiltration. The user's role as VP of Engineering adds critical risk due to access scope. Immediate investigation and revocation of suspicious sessions and app permissions are warranted. Overall, this is a high-risk incident likely indicating account compromise.

🕵️ Threat Intelligence:
IP 45.77.88.99 geolocates to Mumbai, India, a region frequently used by threat actors for proxying. The rapid travel time between San Francisco and Mumbai is impossible, indicating potential account compromise. The us

In [7]:
# --- 5.3 Handoff Orchestration: SOC Alert Routing ---
# A triage agent routes alerts to the right specialist based on alert type.
# Specialists have function tools for their domain actions.
# NOTE: Handoff uses Human-in-the-Loop (HIL) which is incompatible with DevUI.
#       We run a scripted demo here.

from agent_framework import tool
from agent_framework.orchestrations import HandoffAgentUserRequest

print("=" * 60)
print("HANDOFF: SOC Triage → [CompromiseSpecialist, PhishingSpecialist, InsiderThreatSpecialist]")
print("=" * 60)

# --- Specialist function tools ---

@tool(approval_mode="never_require")
def revoke_user_sessions(upn: Annotated[str, "User principal name to revoke sessions for"]) -> str:
    """Simulated function to revoke all active sessions for a compromised user."""
    return f"All active sessions revoked for {upn}. User will need to re-authenticate."

@tool(approval_mode="never_require")
def block_malicious_app(app_name: Annotated[str, "Name of the malicious application to block"]) -> str:
    """Simulated function to block a malicious OAuth application and revoke its consent."""
    return f"Application '{app_name}' has been blocked and all consent grants revoked across the tenant."

@tool(approval_mode="never_require")
def escalate_to_hr(upn: Annotated[str, "User principal name to escalate to HR"]) -> str:
    """Simulated function to escalate a potential insider threat case to HR."""
    return f"Insider threat case escalated to HR for {upn}. Case ID: IT-2025-0042."

# --- Agents ---

soc_triage_agent = Agent(
    client=af_client, name="soc_triage_agent",
    instructions=(
        "You are a SOC L1 triage analyst. Route security alerts to the appropriate specialist:\n"
        "- Account compromise / impossible travel / brute force → compromise_specialist\n"
        "- OAuth phishing / consent grants / suspicious apps → phishing_specialist\n"
        "- Unusual data access patterns / policy violations → insider_threat_specialist\n"
        "Always explain why you're routing to a specific specialist."
    ),
    require_per_service_call_history_persistence=True,
)

compromise_specialist = Agent(
    client=af_client, name="compromise_specialist",
    instructions="You handle account compromise incidents. Investigate and remediate using revoke_user_sessions.",
    tools=[revoke_user_sessions],
    require_per_service_call_history_persistence=True,
)

phishing_specialist = Agent(
    client=af_client, name="phishing_specialist",
    instructions="You handle OAuth phishing and consent phishing attacks. Block malicious apps using block_malicious_app.",
    tools=[block_malicious_app],
    require_per_service_call_history_persistence=True,
)

insider_threat_specialist = Agent(
    client=af_client, name="insider_threat_specialist",
    instructions="You investigate potential insider threats and data exfiltration. Escalate to HR using escalate_to_hr when warranted.",
    tools=[escalate_to_hr],
    require_per_service_call_history_persistence=True,
)

# --- Build handoff workflow ---
handoff_workflow = (
    HandoffBuilder(
        name="soc_alert_routing",
        participants=[soc_triage_agent, compromise_specialist, phishing_specialist, insider_threat_specialist],
        checkpoint_storage=InMemoryCheckpointStorage(),
    )
    .with_start_agent(soc_triage_agent)
    .build()
)
print("✅ Handoff workflow built: soc_triage → [compromise, phishing, insider_threat]")

# --- Run a scripted demo ---
scripted_responses = [
    "We detected an impossible travel alert for john.doe@contoso.com — sign-in from Seattle then Lagos 17 minutes later. The Lagos IP is flagged as malicious.",
    "Yes, please revoke his sessions and reset his password.",
    "Thanks, that's all for now.",
]

def print_events(events):
    """Process events and return pending HIL requests."""
    pending = []
    for event in events:
        if event.type == "handoff_sent":
            print(f"\n[🔀 Handoff: {event.data.source} → {event.data.target}]")
        elif event.type == "output":
            data = event.data
            if isinstance(data, AgentResponse):
                for m in data.messages:
                    if m.text:
                        print(f"- {m.author_name or m.role}: {m.text}")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            for m in event.data.agent_response.messages:
                if m.text:
                    print(f"- {m.author_name or m.role}: {m.text}")
            pending.append(event)
    return pending

print("\n[Starting SOC handoff demo...]\n")
initial_message = "I need help investigating a security alert."
print(f"- User: {initial_message}")

workflow_result = handoff_workflow.run(initial_message, stream=True)
events = [event async for event in workflow_result]
pending_requests = print_events(events)

while pending_requests and scripted_responses:
    user_response = scripted_responses.pop(0)
    print(f"\n- User: {user_response}")
    responses = {
        req.request_id: HandoffAgentUserRequest.create_response(user_response)
        for req in pending_requests
    }
    new_events = await handoff_workflow.run(responses=responses)
    pending_requests = print_events(new_events)

print("\n✅ Handoff orchestration complete")

HANDOFF: SOC Triage → [CompromiseSpecialist, PhishingSpecialist, InsiderThreatSpecialist]
✅ Handoff workflow built: soc_triage → [compromise, phishing, insider_threat]

[Starting SOC handoff demo...]

- User: I need help investigating a security alert.
- soc_triage_agent: Could you please provide more details about the security alert? For example, what type of suspicious activity or behavior has been detected? This will help me determine the appropriate specialist to route the case to.

- User: We detected an impossible travel alert for john.doe@contoso.com — sign-in from Seattle then Lagos 17 minutes later. The Lagos IP is flagged as malicious.

[🔀 Handoff: soc_triage_agent → compromise_specialist]
- compromise_specialist: An impossible travel alert combined with a malicious IP is a strong indicator of a compromised account. I recommend immediately revoking all active sessions for john.doe@contoso.com to prevent further unauthorized access. 

I will proceed with revoking the sessions 

In [11]:
# --- 5.4 Group Chat Orchestration: SOC War Room ---
# An orchestrator coordinates multiple SOC analysts in a structured discussion.

print("=" * 60)
print("GROUP CHAT: SOC War Room — Orchestrator + ThreatHunter + IncidentResponder + ForensicAnalyst")
print("=" * 60)

# Define the specialist agents for the SOC war room
threat_hunter = Agent(
    client=af_client, name="ThreatHunter",
    instructions=(
        "You are a threat hunter. Analyze the alert for indicators of compromise, "
        "map to known threat actor TTPs, and identify potential lateral movement paths. "
        "Be specific about MITRE ATT&CK techniques. Keep under 200 words."
    ),
)

incident_responder = Agent(
    client=af_client, name="IncidentResponder",
    instructions=(
        "You are an incident responder. Based on the threat analysis, propose "
        "immediate containment actions, remediation steps, and recovery procedures. "
        "Prioritize actions by urgency. Keep under 250 words."
    ),
)

forensic_analyst = Agent(
    client=af_client, name="ForensicAnalyst",
    instructions=(
        "You are a digital forensic analyst. Review the proposed response plan for "
        "evidence preservation concerns, suggest additional data collection, and "
        "identify potential gaps in the investigation. Provide 3-5 specific recommendations. "
        "Be direct and concise."
    ),
)

war_room_orchestrator = Agent(
    client=af_client, name="WarRoomOrchestrator",
    description="Coordinates SOC war room discussion by selecting speakers",
    instructions=(
        "You coordinate a SOC war room discussion. Select the best next speaker: "
        "ThreatHunter for IOC analysis and TTPs, IncidentResponder for containment/remediation, "
        "ForensicAnalyst for evidence review. Aim for a productive 3-round discussion."
    ),
)

gc_workflow = (
    GroupChatBuilder(
        participants=[threat_hunter, incident_responder, forensic_analyst],
        orchestrator_agent=war_room_orchestrator,
        termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
        max_rounds=6,
    )
    .build()
)

# Run the workflow directly (avoids as_agent() message-threading issue)
last_agent = None
async for event in gc_workflow.run(
    "CRITICAL ALERT: Disabled account ex.employee@contoso.com (former IT admin) successfully signed in "
    "from Bucharest, Romania at 01:00 AM. The account was disabled 3 months ago. Sign-in used a valid "
    "refresh token. Post-login activity shows access to SharePoint admin center and Azure AD portal.",
    stream=True,
):
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        if event.data.author_name != last_agent:
            last_agent = event.data.author_name
            print(f"\n\n[{last_agent}]:", end=" ")
        print(event.data.text, end="", flush=True)

print("\n\n✅ Group chat orchestration complete")

GROUP CHAT: SOC War Room — Orchestrator + ThreatHunter + IncidentResponder + ForensicAnalyst


[WarRoomOrchestrator]: The team should proceed with the outlined remediation and forensic steps promptly to secure the environment and prevent further compromise.

✅ Group chat orchestration complete


In [ ]:
# --- 5.6 DevUI: Interactive SOC Agent & Workflow Dashboard ---
# Launch DevUI to interactively test all SOC agents and workflows.
# All entities accept simple text input — type your alert description!
#
# NOTE: We build FRESH workflow instances here. Reusing workflows that already
# ran in the notebook would fail with "Queue is bound to a different event loop"
# because their internal asyncio queues are tied to the notebook's event loop.

from agent_framework.devui import serve
import threading

# --- Text input receiver: accepts plain str, forwards as AgentExecutorRequest ---
class TextInput(Executor):
    """Converts plain text input to AgentExecutorRequest so DevUI shows Simple Text."""
    @handler
    async def receive(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        await ctx.send_message(AgentExecutorRequest(
            messages=[Message("user", contents=[prompt])], should_respond=True))

# --- Rebuild fresh workflow instances for DevUI ---

# Fresh Sequential workflow (with text input receiver at the front)
devui_seq_input = TextInput(id="seq-input")
devui_seq_triager = Agent(
    client=af_client, name="Triager",
    instructions=(
        "You are a SOC L1 triage analyst. Given a security alert description, "
        "classify it by type (Impossible Travel, Brute Force, MFA Fatigue, OAuth Phishing, etc.), "
        "assign a severity (Critical/High/Medium/Low), and list the key indicators of compromise. "
        "Keep output concise — under 150 words."
    ),
)
devui_seq_enricher = Agent(
    client=af_client, name="Enricher",
    instructions=(
        "You are a SOC enrichment specialist. Given a triaged alert, add context: "
        "map to MITRE ATT&CK techniques, estimate the travel velocity if locations are given, "
        "assess IP reputation indicators, and note any privilege escalation risk. "
        "Output only the enriched context — under 200 words."
    ),
)
devui_seq_reporter = Agent(
    client=af_client, name="ReportWriter",
    instructions=(
        "You are a SOC report writer. Given enriched alert context, produce a concise "
        "executive-ready incident report with: Summary, Risk Level, MITRE Techniques, "
        "Verdict (PACO framework), and Recommended Actions. Use markdown formatting. "
        "Keep under 300 words."
    ),
)
devui_seq_workflow = (
    WorkflowBuilder(name="SOC Alert Triage Pipeline", start_executor=devui_seq_input)
    .add_edge(devui_seq_input, devui_seq_triager)
    .add_chain([devui_seq_triager, devui_seq_enricher, devui_seq_reporter])
    .build()
)

# Fresh Concurrent workflow (DispatchToAnalysts already accepts str — Simple Text)
devui_dispatcher = DispatchToAnalysts(id="dispatcher")
devui_aggregator = AggregateAssessments(id="aggregator")
devui_expert_identity = AgentExecutor(
    Agent(client=af_client, name="identity_analyst",
          instructions=(
              "You are an identity security analyst. Evaluate the alert for: authentication anomalies, "
              "MFA status, device compliance, session tokens, and user risk state. "
              "Provide a concise risk assessment under 150 words."
          ))
)
devui_expert_threat_intel = AgentExecutor(
    Agent(client=af_client, name="threat_intel_analyst",
          instructions=(
              "You are a threat intelligence analyst. Evaluate IP reputation, geolocation anomalies, "
              "known threat actor TTPs, and MITRE ATT&CK mapping. "
              "Provide a concise assessment under 150 words."
          ))
)
devui_expert_compliance = AgentExecutor(
    Agent(client=af_client, name="compliance_analyst",
          instructions=(
              "You are a compliance and policy analyst. Assess the alert against organizational "
              "security policies: SLA adherence, data access risk, regulatory implications (GDPR, SOX), "
              "and required escalation paths. Provide a concise assessment under 150 words."
          ))
)
devui_conc_workflow = (
    WorkflowBuilder(name="SOC Parallel Risk Assessment", start_executor=devui_dispatcher)
    .add_fan_out_edges(devui_dispatcher, [devui_expert_identity, devui_expert_threat_intel, devui_expert_compliance])
    .add_fan_in_edges([devui_expert_identity, devui_expert_threat_intel, devui_expert_compliance], devui_aggregator)
    .build()
)

# Fresh GroupChat agents & workflow
devui_threat_hunter = Agent(
    client=af_client, name="ThreatHunter",
    instructions=(
        "You are a threat hunter. Analyze the alert for indicators of compromise, "
        "map to known threat actor TTPs, and identify potential lateral movement paths. "
        "Be specific about MITRE ATT&CK techniques. Keep under 200 words."
    ),
)
devui_incident_responder = Agent(
    client=af_client, name="IncidentResponder",
    instructions=(
        "You are an incident responder. Based on the threat analysis, propose "
        "immediate containment actions, remediation steps, and recovery procedures. "
        "Prioritize actions by urgency. Keep under 250 words."
    ),
)
devui_forensic_analyst = Agent(
    client=af_client, name="ForensicAnalyst",
    instructions=(
        "You are a digital forensic analyst. Review the proposed response plan for "
        "evidence preservation concerns, suggest additional data collection, and "
        "identify potential gaps in the investigation. Provide 3-5 specific recommendations. "
        "Be direct and concise."
    ),
)
devui_war_room_orchestrator = Agent(
    client=af_client, name="WarRoomOrchestrator",
    description="Coordinates SOC war room discussion by selecting speakers",
    instructions=(
        "You coordinate a SOC war room discussion. Select the best next speaker: "
        "ThreatHunter for IOC analysis and TTPs, IncidentResponder for containment/remediation, "
        "ForensicAnalyst for evidence review. Aim for a productive 3-round discussion."
    ),
)

# --- Collect entities ---
entities = [
    # Individual agents — chat interface
    devui_threat_hunter,
    devui_incident_responder,
    devui_forensic_analyst,
    # Workflows — graph visualization + Simple Text input
    devui_seq_workflow,
    devui_conc_workflow,
    # GroupChat
    GroupChatBuilder(
        participants=[devui_threat_hunter, devui_incident_responder, devui_forensic_analyst],
        orchestrator_agent=devui_war_room_orchestrator,
        termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
        max_rounds=6,
    ).build().as_agent(name="SOC War Room Discussion"),
]

print("🚀 Starting DevUI with all SOC agents and workflows...")
devui_thread = threading.Thread(
    target=serve,
    kwargs={
        "entities": entities,
        "port": 8070,
        "auto_open": False,
        "instrumentation_enabled": True,
        "auth_enabled": False,
    },
    daemon=True,
)
devui_thread.start()
import time; time.sleep(2)

print("✅ DevUI running at: http://localhost:8070")
print(f"\n   Loaded {len(entities)} entities")
print("\n   All entities accept simple text input — describe your security alert!")
print("   Workflows show the node graph with execution timeline.")
print("\n   NOTE: Handoff workflow uses HIL which is not yet supported in DevUI.")
print("         Test it by running the scripted demo in cell 8 (Section 5.3).")

C:\Users\haozhang2\AppData\Local\Temp\ipykernel_24928\1811447494.py:55: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()
C:\Users\haozhang2\AppData\Local\Temp\ipykernel_24928\1811447494.py:89: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()


🚀 Starting DevUI with all SOC agents and workflows...


INFO:     Started server process [24928]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8070 (Press CTRL+C to quit)


✅ DevUI running at: http://localhost:8070

   Loaded 6 entities

   All entities accept simple text input — describe your security alert!
   Workflows show the node graph with execution timeline.

   NOTE: Handoff workflow uses HIL which is not yet supported in DevUI.
         Test it by running the scripted demo in cell 8 (Section 5.3).


INFO:     127.0.0.1:64520 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64520 - "GET /assets/index.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "GET /assets/index.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:64520 - "GET /meta HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "GET /agentframework.svg HTTP/1.1" 200 OK
INFO:     127.0.0.1:64520 - "GET /v1/entities HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "GET /v1/conversations?agent_id=agent_in_memory_threathunter_6e9b05ed36fa46c19a7d06f0a12e18ca HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "GET /v1/entities/workflow_in_memory_soc-alert-triage-pipeline_73bd61cc99df4021967a11de8e45bbda/info?type=workflow HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "GET /v1/conversations?entity_id=workflow_in_memory_soc-alert-triage-pipeline_73bd61cc99df4021967a11de8e45bbda&type=workflow_session HTTP/1.1" 200 OK
INFO:     127.0.0.1:62210 - "POST /v1/conversations HTTP/1.1" 200 OK
INFO:    

> 💡 **Check the traces**: Open **Foundry Portal → Tracing**. You'll see server-side traces
> for each agent invocation, showing tool calls, model inputs/outputs, and orchestration flow.

---

---
## Next Steps

You've completed **Foundry Agent Basics — SOC Edition!**

Now apply these concepts to a full impossible travel investigation workflow:

1. **`multi_agent/`** — Fan-out/fan-in orchestration with 10 specialized risk-analysis agents and deterministic aggregation. Start with `00-setup.ipynb`.
2. **`single_agent_skill/`** — A single agent augmented with a file-based skill package containing domain guidance and helper scripts. Start with `00-setup.ipynb`.

Both implementations use the same five synthetic detection scenarios and return a structured `InvestigationVerdict`.